In [1]:
import json 

with open('F:\personal\Paper\毕业论文\Data\COSV\cosv1.json','r') as f:
    data = json.load(f)
    
data = data[:1000]

In [5]:
deepseek_key = 'sk-4c51074aea8d45e7ad24119862e0b626'
from zhipuai import ZhipuAI
client = ZhipuAI(api_key="1c697a8c2cd74186810193bab3c8d2bb.8PJZRTwbMn7DFyld") # 填写您自己的APIKey
def get_response(text):
    response = client.chat.completions.create(
        model="glm-4",  # 填写需要调用的模型编码
        messages=[
            {"role": "system", "content":f"""
        请从以下网络安全漏洞描述中抽取结构化知识。按JSON格式返回包含以下内容的三元组列表：
        {{
        "triples": [
            {{
            "head": "实体或概念",
            "relation": "关系类型",
            "tail": "关联实体或概念"
            }}
        ]
        }}
        直接输出三元组列表，不需要包含其他内容。
        可用关系类型：
        - 漏洞影响
        - 存在于版本
        - 属于分类
        - 具有风险等级
        - 修复版本
        - 有别名
            等"""},
            {"role": "user", "content": f"待处理数据{text}"}
        ],
    )
    return response.choices[0].message.content

In [37]:
res = get_response(data[1])
res = json.loads(res)
for i in range(2, 100):
    subres = get_response(data[i])
    for j in json.loads(subres):
        res.append(j)

In [38]:
print(res)
print(len(res))
print(res[3])

[{'head': 'CNNVD-202312-2285', 'relation': '有别名', 'tail': 'CVE-2023-46681'}, {'head': 'CNNVD-202312-2285', 'relation': '存在于版本', 'tail': '2.37及之前版本'}, {'head': 'CNNVD-202312-2285', 'relation': '漏洞影响', 'tail': '允许经过身份验证的攻击者访问产品的命令行接口来执行任意命令'}, {'head': 'CNNVD-202312-2285', 'relation': '具有风险等级', 'tail': '高危'}, {'head': 'CNNVD-202312-1777', 'relation': '有别名', 'tail': 'CVE-2023-6928'}, {'head': 'CNNVD-202312-1777', 'relation': '存在于版本', 'tail': 'EuroTel ETL3100 v01c01'}, {'head': 'CNNVD-202312-1777', 'relation': '存在于版本', 'tail': 'EuroTel ETL3100 v01x37'}, {'head': 'CNNVD-202312-1777', 'relation': '具有风险等级', 'tail': '超危'}, {'head': 'CNNVD-202312-2327', 'relation': '有别名', 'tail': 'CVE-2023-5645'}, {'head': 'CNNVD-202312-2327', 'relation': '存在于版本', 'tail': 'WP Mail Log 1.1.3'}, {'head': 'CNNVD-202312-2327', 'relation': '具有风险等级', 'tail': '中危'}, {'head': 'CNNVD-202312-2322', 'relation': '有别名', 'tail': 'CVE-2023-5931'}, {'head': 'CNNVD-202312-2322', 'relation': '存在于版本', 'tail': 'WordPress plugin rt

In [41]:
from neo4j import GraphDatabase

class Neo4jLoader:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        
    def load_triples(self, data):
        with self.driver.session() as session:
            for triple in data:
                #print(triple)
                session.execute_write(
                    self._create_triple,
                    triple["head"],
                    triple["relation"],
                    triple["tail"]
                )
                
    @staticmethod
    def _create_triple(tx, head, relation, tail):
        query = """
        MERGE (h:Entity {name: $head})
        MERGE (t:Entity {name: $tail})
        MERGE (h)-[r:RELATION {type: $relation}]->(t)
        """
        tx.run(query, head=head, relation=relation, tail=tail)

# 初始化连接
loader = Neo4jLoader("bolt://localhost:7687", "neo4j", "MnV8mXmgdhYp-F-")
#print(json.dumps(res, indent=2, ensure_ascii=False))
#result = json.loads(res)
loader.load_triples(res)